In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
# 【关键两行】——必须同时设置！
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'PingFang SC', 'Heiti TC', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题[1][2][3][4][5][6][7][8][9][10]

import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/聚宽数据')
import datasource 
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.getAStockCodes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:4938,第一支股票:000001.XSHE


index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-07-06      0  6.45  6.58  6.33   6.49  8.520715e+08   
            2018-07-09      1  6.51  6.76  6.50   6.76  1.255008e+09   
            2018-07-10      2  6.76  6.76  6.66   6.73  8.030390e+08   
            2018-07-11      3  6.56  6.61  6.50   6.58  7.447658e+08   
            2018-07-12      4  6.54  6.82  6.52   6.75  1.008659e+09   

                             volume  
code        date                     
000001.XSHE 2018-07-06  131957580.0  
            2018-07-09  188260099.0  
            2018-07-10  119750900.0  
            2018-07-11  113666923.0  
            2018-07-12  149984167.0

In [2]:
# macd的3个参数
fastperiod=12
slowperiod=26
signalperiod=9
# 如下是macd，分别是dif， signal(dea)，hist
dt['macd_dif'], dt['macd_signal'], dt['macdhist'] = talib.MACD(dt['close'], fastperiod=fastperiod, slowperiod=slowperiod, signalperiod=signalperiod)
dt['pre_macd_dif'] = dt['macd_dif'].shift(1)
dt['pre_macd_signal'] = dt['macd_signal'].shift(1)
# 未来几天的收盘价
for i in range(5):
    dt[f'nextClose{i+1}'] = dt['close'].shift(-(i+1))
dt.iloc[50:60].head()

index  open  high   low  close         money  \
code        date                                                       
000001.XSHE 2018-09-14     50  7.59  7.59  7.46   7.48  6.114225e+08   
            2018-09-17     51  7.45  7.45  7.36   7.37  5.026096e+08   
            2018-09-18     52  7.40  7.70  7.38   7.66  1.120868e+09   
            2018-09-19     53  7.63  7.86  7.63   7.79  1.257192e+09   
            2018-09-20     54  7.81  7.88  7.75   7.78  6.367428e+08   

                             volume  macd_dif  macd_signal  macdhist  \
code        date                                                       
000001.XSHE 2018-09-14   81502274.0  0.134937     0.179134 -0.044197   
            2018-09-17   68000601.0  0.110775     0.165462 -0.054687   
            2018-09-18  148184491.0  0.113716     0.155113 -0.041397   
            2018-09-19  161958845.0  0.125095     0.149109 -0.024014   
            2018-09-20   81580938.0  0.131787     0.145645 -0.013858   

                        pre_macd_dif  pre_macd_signal  nextClose1  nextClose2  \
code        date                                                                
000001.XSHE 2018-09-14      0.152614         0.190183        7.37        7.66   
            2018-09-17      0.134937         0.179134        7.66        7.79   
            2018-09-18      0.110775         0.165462        7.79        7.78   
            2018-09-19      0.113716         0.155113        7.78        8.11   
            2018-09-20      0.125095         0.149109        8.11        8.02   

                        nextClose3  nextClose4  nextClose5  
code        date                                            
000001.XSHE 2018-09-14        7.79        7.78        8.11  
            2018-09-17        7.78        8.11        8.02  
            2018-09-18        8.11        8.02        8.14  
            2018-09-19        8.02        8.14        8.17  
            2018-09-20        8.14        8.17        8.40

In [5]:
# 这里先看小于0的时候
dt2 = dt.loc[(dt['macd_dif'] < 0 ) & (dt['macd_signal'] < 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()
# 这里看看比值
for i in range(5):
    dt2[f'rate{i+1}'] = (dt2[f'nextClose{i+1}']-dt2['close'])/dt2['close']*100
    # 这里看一下结果把
    _avg = dt2[f'rate{i+1}'].mean()
    _median = dt2[f'rate{i+1}'].median()
    print(f'{i+1}平均值:{_avg:.1f}, 中位值:{_median:.1f}')

1平均值:0.3, 中位值:-0.1
2平均值:0.6, 中位值:0.0
3平均值:0.6, 中位值:-0.3
4平均值:0.9, 中位值:0.4
5平均值:0.8, 中位值:0.4


In [6]:
# 这里看大于0的时候
dt3 = dt.loc[(dt['macd_dif'] > 0 ) & (dt['macd_signal'] > 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()
# 这里看看比值
for i in range(5):
    dt3[f'rate{i+1}'] = (dt3[f'nextClose{i+1}']-dt3['close'])/dt3['close']*100
    # 这里看一下结果把
    _avg = dt3[f'rate{i+1}'].mean()
    _median = dt3[f'rate{i+1}'].median()
    print(f'{i+1}平均值:{_avg:.1f}, 中位值:{_median:.1f}')

1平均值:0.6, 中位值:0.5
2平均值:0.9, 中位值:0.4
3平均值:1.3, 中位值:0.9
4平均值:1.6, 中位值:1.1
5平均值:1.1, 中位值:1.8


In [7]:
# 看起来像是大于0的情况下，股票更强势

In [8]:
# 这里看看全部股票
lst_macd_lt_0 = []
lst_macd_gt_0 = []
# macd的3个参数
fastperiod=12
slowperiod=26
signalperiod=9
# 对全部股票进行
for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.getData(code)
    # 如下是macd，分别是dif， signal(dea)，hist
    dt['macd_dif'], dt['macd_signal'], dt['macdhist'] = talib.MACD(dt['close'], fastperiod=fastperiod, slowperiod=slowperiod, signalperiod=signalperiod)
    dt['pre_macd_dif'] = dt['macd_dif'].shift(1)
    dt['pre_macd_signal'] = dt['macd_signal'].shift(1)
    # 未来几天的收盘价
    for i in range(5):
        dt[f'nextClose{i+1}'] = dt['close'].shift(-(i+1))
    # 筛选
    dt2 = dt.loc[(dt['macd_dif'] < 0 ) & (dt['macd_signal'] < 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()
    if len(dt2) > 0:
        lst_macd_lt_0.append(dt2)
    dt3 = dt.loc[(dt['macd_dif'] > 0 ) & (dt['macd_signal'] > 0 ) & (dt['macd_dif'] > dt['macd_signal'] ) & (dt['pre_macd_dif'] < dt['pre_macd_signal'])].dropna()
    if len(dt3) > 0:
        lst_macd_gt_0.append(dt3)

100%|██████████████████████████████████████████████████████████████████████████████| 4938/4938 [01:55<00:00, 42.88it/s]


In [9]:
# 拼接
dt_macd_lt_0 = pd.concat(lst_macd_lt_0)
dt_macd_gt_0 = pd.concat(lst_macd_gt_0)
print('macd小于0')
for i in range(5):
    dt_macd_lt_0[f'rate{i+1}'] = (dt_macd_lt_0[f'nextClose{i+1}']-dt_macd_lt_0['close'])/dt_macd_lt_0['close']*100
    # 这里看一下结果把
    _avg = dt_macd_lt_0[f'rate{i+1}'].mean()
    _median = dt_macd_lt_0[f'rate{i+1}'].median()
    print(f'{i+1}平均值:{_avg:.1f}, 中位值:{_median:.1f}')
print('macd大于0')
for i in range(5):
    dt_macd_gt_0[f'rate{i+1}'] = (dt_macd_gt_0[f'nextClose{i+1}']-dt_macd_gt_0['close'])/dt_macd_gt_0['close']*100
    # 这里看一下结果把
    _avg = dt_macd_gt_0[f'rate{i+1}'].mean()
    _median = dt_macd_gt_0[f'rate{i+1}'].median()
    print(f'{i+1}平均值:{_avg:.1f}, 中位值:{_median:.1f}')


macd小于0
1平均值:0.2, 中位值:0.0
2平均值:0.3, 中位值:0.0
3平均值:0.4, 中位值:0.0
4平均值:0.4, 中位值:-0.1
5平均值:0.5, 中位值:-0.1
macd大于0
1平均值:0.2, 中位值:-0.2
2平均值:0.2, 中位值:-0.3
3平均值:0.2, 中位值:-0.3
4平均值:0.2, 中位值:-0.4
5平均值:0.2, 中位值:-0.6


但总体上来说，macd大于0反倒是弱势。